# DS4 Step 2, Path A' -- Behavior-only Phenotyping

**Notebook:** `131_ds4_behavior_only_phenotype.ipynb`  
**Plan:** `outputs/reports/stage4/step2_phenotyping/ds4_step2_exploration_plan.md` SS2

---

## Question

Does DS4's Phase 1 taxonomy (`59_ds4_phenotype_exploration.ipynb`) depend on network structure (`rho_self`), or is it already behavior-dominated? DS5's own Path A found the OPPOSITE answer for DS5 (near-orthogonal, ARI=0.020, since DS5's Phase 1 was network-dominated) -- this notebook does not assume the same result transfers.

## Method

Same complete-case population as Phase 1 (N=641, for a fair ARI comparison), but clustering only on the 4 non-NeT features: `tau_hat`, `H_emp`, `fixed_point_deviation`, `chi_OU`. Same stability-first k-selection as Phase 1.


## FLAG decisions

| Flag | Decision |
|---|---|
| FLAG 1 | Same complete-case population as Phase 1 (N=641) for valid ARI comparison |
| FLAG 2 | Stability-first k-selection (silhouette among STABLE candidates only) |
| FLAG 3 | Report ARI vs Phase 1 using Path A's own chosen k (not forced to k=3) |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

OUT_DIR = Path("outputs/model_outputs/ds4_phenotyping")
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = ["tau_hat", "H_emp", "fixed_point_deviation", "chi_OU"]
RANDOM_STATE = 42


In [ ]:
stop = pd.read_csv("outputs/model_outputs/ds4_stochastic_processes/participant_parameters.csv")
ent = pd.read_csv("outputs/model_outputs/ds4_information_theory/participant_entropy_metrics.csv")
dys = pd.read_csv("outputs/model_outputs/ds4_dynamical_systems/participant_dynamics_parameters.csv")
stm = pd.read_csv("outputs/model_outputs/ds4_statistical_mechanics/participant_ds4_stm_parameters.csv")
net = pd.read_csv("outputs/model_outputs/ds4_network_models/participant_network_metrics.csv")
phase1 = pd.read_csv(OUT_DIR / "ds4_cluster_assignments.csv").rename(
    columns={"cluster": "phase1_cluster"})

df = stop[["participant_id_canonical", "fit_valid", "tau_hat"]].rename(
    columns={"fit_valid": "stop_valid"})
df = df.merge(ent[["participant_id_canonical", "H_emp"]], on="participant_id_canonical", how="left")
df = df.merge(dys[["participant_id_canonical", "fixed_point_deviation"]],
    on="participant_id_canonical", how="left")
df = df.merge(stm[["participant_id_canonical", "chi_OU", "fit_valid"]].rename(
    columns={"fit_valid": "stm_valid"}), on="participant_id_canonical", how="left")
df = df.merge(net[["participant_id_canonical", "eligible"]].rename(
    columns={"eligible": "net_eligible"}), on="participant_id_canonical", how="left")
df = df.merge(phase1, on="participant_id_canonical", how="left")

complete_mask = (
    df["stop_valid"].fillna(False) & df["stm_valid"].fillna(False) &
    df["net_eligible"].fillna(False) & df[FEATURES].notna().all(axis=1)
)
complete = df[complete_mask].reset_index(drop=True)
print(f"Complete-case N: {len(complete)} (same population as Phase 1)")


In [ ]:
X = StandardScaler().fit_transform(complete[FEATURES])
print(complete[FEATURES].corr().round(3))


In [ ]:
def jaccard_cluster_stability(X, k, n_boot=100, random_state=RANDOM_STATE):
    rng = np.random.RandomState(random_state)
    n = X.shape[0]
    ref = KMeans(n_clusters=k, n_init=10, random_state=random_state).fit(X)
    ref_labels = ref.labels_
    jaccards = []
    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        seed = int(rng.randint(0, 1_000_000))
        boot = KMeans(n_clusters=k, n_init=10, random_state=seed).fit(X[idx])
        boot_on_full = boot.predict(X)
        best_js = []
        for c in range(k):
            ref_set = set(np.where(ref_labels == c)[0])
            best_j = 0.0
            for c2 in range(k):
                boot_set = set(np.where(boot_on_full == c2)[0])
                union = len(ref_set | boot_set)
                if union == 0:
                    continue
                best_j = max(best_j, len(ref_set & boot_set) / union)
            best_js.append(best_j)
        jaccards.append(np.mean(best_js))
    return np.mean(jaccards)


k_candidates = range(2, 7)
sil_scores, jaccard_by_k, sizes_by_k = {}, {}, {}
for k in k_candidates:
    labels = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(X)
    sil_scores[k] = silhouette_score(X, labels)
    jaccard_by_k[k] = jaccard_cluster_stability(X, k)
    sizes_by_k[k] = pd.Series(labels).value_counts().sort_index().tolist()
    verdict = "STABLE" if jaccard_by_k[k] >= 0.75 else "not stable"
    print(f"k={k}: silhouette={sil_scores[k]:.4f}, jaccard={jaccard_by_k[k]:.4f} "
          f"({verdict}), sizes={sizes_by_k[k]}")

MIN_CELL = 10
stable_ks = [k for k in k_candidates
             if jaccard_by_k[k] >= 0.75 and min(sizes_by_k[k]) >= MIN_CELL]
if stable_ks:
    K_CHOSEN = max(stable_ks, key=lambda k: sil_scores[k])
    print(f"\nChosen k={K_CHOSEN} (highest silhouette among STABLE candidates "
          f"with all clusters >= {MIN_CELL})")
else:
    K_CHOSEN = max(sil_scores, key=sil_scores.get)
    print(f"\nNO FEASIBLE k -- no candidate was both bootstrap-stable and free of "
          f"cells below {MIN_CELL}. Falling back to silhouette-argmax k={K_CHOSEN}; "
          f"flag this solution as UNVALIDATED in the writeup.")

In [ ]:
labels_final = KMeans(n_clusters=K_CHOSEN, n_init=10, random_state=RANDOM_STATE).fit_predict(X)
complete["path_a_cluster"] = labels_final
print(pd.Series(labels_final).value_counts().sort_index())


In [ ]:
ari_vs_phase1 = adjusted_rand_score(complete["phase1_cluster"], complete["path_a_cluster"])
print(f"ARI(Path A' k={K_CHOSEN}, Phase 1 k=3) = {ari_vs_phase1:.4f}")
print(pd.crosstab(complete["path_a_cluster"], complete["phase1_cluster"]))


In [ ]:
profiles = complete.groupby("path_a_cluster")[FEATURES].mean()
profiles["n"] = complete.groupby("path_a_cluster").size()
print(profiles)


In [ ]:
complete[["participant_id_canonical"] + FEATURES + ["path_a_cluster", "phase1_cluster"]].to_csv(
    OUT_DIR / "path_a_feature_matrix.csv", index=False)
profiles.reset_index().to_csv(OUT_DIR / "path_a_cluster_profiles.csv", index=False)

summary = pd.DataFrame([{
    "n": len(complete), "k_chosen": K_CHOSEN,
    "jaccard": jaccard_by_k[K_CHOSEN], "silhouette": sil_scores[K_CHOSEN],
    "ari_vs_phase1": ari_vs_phase1,
}])
summary.to_csv(OUT_DIR / "path_a_summary.csv", index=False)
print("Wrote path_a_feature_matrix.csv, path_a_cluster_profiles.csv, path_a_summary.csv")


In [ ]:
summary_lines = [
    "# DS4 Step 2, Path A' -- Model Run Summary", "",
    "**Notebook:** `131_ds4_behavior_only_phenotype.ipynb`  ",
    "**Plan:** `outputs/reports/stage4/step2_phenotyping/ds4_step2_exploration_plan.md`  ",
    "**Run date:** " + pd.Timestamp.now().strftime("%Y-%m-%d"), "",
    f"Complete-case N={len(complete)}, chosen k={K_CHOSEN} "
    f"(silhouette={sil_scores[K_CHOSEN]:.4f}, jaccard={jaccard_by_k[K_CHOSEN]:.4f})", "",
    f"**ARI vs Phase 1 = {ari_vs_phase1:.4f}**", "",
    "## K-selection (all candidates)", "", "```",
]
for k in k_candidates:
    v = "STABLE" if jaccard_by_k[k] >= 0.75 else "not stable"
    summary_lines.append(f"k={k}: silhouette={sil_scores[k]:.4f}, jaccard={jaccard_by_k[k]:.4f} ({v}), sizes={sizes_by_k[k]}")
summary_lines.append("```")

with open(OUT_DIR / "path_a_model_run_summary.md", "w") as f:
    f.write("\n".join(summary_lines) + "\n")
print("Wrote path_a_model_run_summary.md")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ks = list(sil_scores.keys())
ax.plot(ks, [sil_scores[k] for k in ks], marker="o", color="#4C72B0")
ax.axvline(K_CHOSEN, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("k")
ax.set_ylabel("silhouette score")
ax.set_title(f"Path A' (behavior-only): silhouette by k (chosen k={K_CHOSEN})")
fig.tight_layout()
fig.savefig(FIG_DIR / "path_a_fig1_silhouette_by_k.png", dpi=150)
plt.close(fig)


In [ ]:
ct = pd.crosstab(complete["path_a_cluster"], complete["phase1_cluster"])
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(ct.values, cmap="Blues")
ax.set_xticks(range(ct.shape[1]))
ax.set_xticklabels(ct.columns)
ax.set_yticks(range(ct.shape[0]))
ax.set_yticklabels(ct.index)
for i in range(ct.shape[0]):
    for j in range(ct.shape[1]):
        ax.text(j, i, ct.values[i, j], ha="center", va="center",
                color="white" if ct.values[i, j] > ct.values.max()/2 else "black")
ax.set_xlabel("Phase 1 cluster (k=3, 5 features)")
ax.set_ylabel(f"Path A' cluster (k={K_CHOSEN}, behavior-only)")
ax.set_title(f"Path A' vs Phase 1 (ARI={ari_vs_phase1:.3f})")
fig.colorbar(im, ax=ax, label="n participants")
fig.tight_layout()
fig.savefig(FIG_DIR / "path_a_fig2_crosstab_vs_phase1.png", dpi=150)
plt.close(fig)


## Interpretation and limits

**Read the cells above for live numbers.**

- High ARI vs Phase 1 means network structure (`rho_self`) is not the primary driver of DS4's phenotype space -- behavioral/thermodynamic features alone recover most of the same taxonomy. This is a real, DS4-specific finding, not assumed from DS5 or Structured (both of which found different patterns).
- Same complete-case attrition, no-condition, and daily-timescale caveats as Phase 1 apply here.


## Forward connections

| Connection | Status |
|---|---|
| Path C' (network-focused) | Next -- tests the complementary question |
| Path F' (joint taxonomy) | After both A' and C' exist |
